# CAPE-TTA OpenVLA Pilot V3 — 固定版本与完整运行包

本 notebook 修复配置换行、MuJoCo 版本漂移、旧 checkout 复用和报错丢失。
使用固定源代码版本 `86f393928ae264eb000c15e9a1bf2924c32fb47d`，LIBERO 和模型版本会记录在运行包中。

**操作：代码执行程序 → 更改运行时类型 → T4 GPU → 保存，然后运行下面唯一的代码单元格。**

结束后自动下载 ZIP，成功与失败都会下载。请直接把 ZIP 发回对话，不必逐页截图。
`manifest.json` 的 `status=pilot_validated` 仅表示单状态试验完整性检查通过，
不是完整 CAPE-TTA 方法有效，也不是论文主实验完成。

该版本尚未在 GPU 上验证。需要 Colab 的可用 GPU 配额和正常联网；
下载模型及安装依赖可能耗时，页面持续输出日志时不要重复点击运行。


In [ ]:
# V3: fixed source revision; every run has a unique evidence bundle.
import datetime as dt
import json
from pathlib import Path
import shutil
import subprocess
import sys
import traceback
import uuid

SOURCE_SHA = "86f393928ae264eb000c15e9a1bf2924c32fb47d"
SOURCE = Path("/content/cape-source") / SOURCE_SHA
RUN_DIR = Path("/content/cape-results") / ("run-" + dt.datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:8])

def visible(command):
    with subprocess.Popen(list(map(str, command)), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
        status = proc.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)

try:
    if not SOURCE.exists():
        SOURCE.mkdir(parents=True)
        visible(["git", "-C", SOURCE, "init"])
        visible(["git", "-C", SOURCE, "remote", "add", "origin", "https://github.com/weichezhang/CAPE-TTA.git"])
        visible(["git", "-C", SOURCE, "fetch", "--depth", "1", "origin", SOURCE_SHA])
        visible(["git", "-C", SOURCE, "checkout", "--detach", SOURCE_SHA])
    actual = subprocess.check_output(["git", "-C", str(SOURCE), "rev-parse", "HEAD"], text=True).strip()
    if actual != SOURCE_SHA:
        raise RuntimeError("Unexpected source revision: " + actual)
    subprocess.run(["git", "-C", str(SOURCE), "diff", "--quiet", "HEAD"], check=True)
    visible([sys.executable, "-u", SOURCE / "scripts/colab_bootstrap.py", "--run-dir", RUN_DIR])
except BaseException:
    if not RUN_DIR.with_suffix(".zip").exists():
        RUN_DIR.mkdir(parents=True, exist_ok=True)
        (RUN_DIR / "launcher_failure.txt").write_text(traceback.format_exc(), encoding="utf-8")
        shutil.make_archive(str(RUN_DIR), "zip", RUN_DIR)
    raise
finally:
    bundle = RUN_DIR.with_suffix(".zip")
    if bundle.is_file():
        print("\n下载这个 ZIP 发回本对话；成功或失败都保留。", flush=True)
        from google.colab import files
        files.download(str(bundle))
